In [1]:
!pip install -q sentence-transformers weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.2 MB/s eta 0:00:00


In [1]:
import json
from pathlib import Path


In [2]:
CHUNKS_PATH = Path("/content/chunks.json")

In [3]:
with open(CHUNKS_PATH) as f:
    chunks = json.load(f)


In [5]:
print(f"Total chunks loaded: {len(chunks)}")
print(f"\nSample chunk (keys and values) - CONFIRM!:")
sample = chunks[0]
for k, v in sample.items():
    v_display = (str(v)[:80] + "...") if isinstance(v, str) and len(str(v)) > 80 else v
    print(f"  {k}: {v_display}")

Total chunks loaded: 2030

Sample chunk (keys and values) - CONFIRM!:
  chunk_id: blast-management-pdf:6:0
  text: The initial infections occurs on leaves usually around tillering and appear as d...
  source_document: Blast-Management.pdf
  document_title: Rice Blast Disease Management
  organization: LSU AgCenter
  source_url: None
  document_type: bulletin
  ocr_derived: False
  chunk_type: narrative
  disease_name: None
  disease_name_is_canonical: False
  source_disease_label: None
  section: None
  page_start: 7
  page_end: 7
  word_count: 63
  chunk_index: 0
  chunk_total: 1
  is_inoculation_protocol: False
  symptom: None
  recommended_treatment: None
  active_ingredient: None
  dosage: None
  crop_stage: None


In [6]:
CANONICAL_CLASSES = {
    "Bacterial_Leaf_Blight", "Brown_Spot", "Leaf_Blast",
    "Narrow_Brown", "Rice_Tungro", "Sheath_Blight",
}


In [7]:
assert "chunk_id" in sample, "chunk_id missing - required as the stable key for Phase 5"
assert len(chunks) == len({c["chunk_id"] for c in chunks}), "chunk_id is not unique across all chunks!"


In [8]:
disease_names = {c.get("disease_name") for c in chunks if c.get("disease_name")}
unexpected = disease_names - CANONICAL_CLASSES

In [9]:
if unexpected:
    print(f"\nWARNING: disease_name values not in the 6 canonical classes: {unexpected}")
    print("These may be legitimate non-canonical disease names per spec §1 - confirm before proceeding.")


These may be legitimate non-canonical disease names per spec §1 - confirm before proceeding.


In [10]:
null_disease = sum(1 for c in chunks if not c.get("disease_name"))
print(f"\nChunks with disease_name=null: {null_disease} (expect ~921 per Phase 3 report)")


Chunks with disease_name=null: 921 (expect ~921 per Phase 3 report)


In [11]:
null_disease = sum(1 for c in chunks if not c.get("disease_name"))
print(f"\nChunks with disease_name=null: {null_disease} (expect ~921 per Phase 3 report)")


Chunks with disease_name=null: 921 (expect ~921 per Phase 3 report)


In [12]:
inoculation_as_treatment = sum(
    1 for c in chunks
    if c.get("is_inoculation_protocol") and c.get("chunk_type") == "treatment_record"
)

In [13]:
assert inoculation_as_treatment == 0, (
    f"SAFETY INVARIANT VIOLATED: {inoculation_as_treatment} inoculation-protocol "
    f"chunks are tagged treatment_record. Stop and fix before embedding."
)
print(f"\nSafety invariant holds: 0 inoculation-protocol chunks tagged as treatment_record.")



Safety invariant holds: 0 inoculation-protocol chunks tagged as treatment_record.


In [15]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [16]:
model = SentenceTransformer("intfloat/e5-large-v2", device="cuda")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [17]:
texts_prefixed = [f"passage: {c['text']}" for c in chunks]

In [18]:
embeddings = model.encode(
    texts_prefixed,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,  # e5 is trained for cosine similarity on normalized vectors
)


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

In [19]:
print(f"Generated {len(embeddings)} embeddings of dim {embeddings.shape[1]}")


Generated 2030 embeddings of dim 1024


In [21]:
chunk_ids = [c["chunk_id"] for c in chunks]
np.savez("/content/embeddings.npz", chunk_ids=chunk_ids, vectors=embeddings)
print("Saved to /content/embeddings.npz (chunk_ids + vectors, paired by index)")


Saved to /content/embeddings.npz (chunk_ids + vectors, paired by index)


In [22]:
import weaviate

In [23]:
from weaviate.classes.init import Auth
from weaviate.classes.config import Configure, Property, DataType

In [ ]:
WEAVIATE_URL = "https://cloud.weaviate.network"
WEAVIATE_API_KEY = "api-key"
CLASS_NAME = "cluster_name"

In [25]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
)


In [26]:
SCHEMA_PROPERTIES = [
    Property(name="chunk_id", data_type=DataType.TEXT),
    Property(name="text", data_type=DataType.TEXT),
    Property(name="source_document", data_type=DataType.TEXT),
    Property(name="document_title", data_type=DataType.TEXT),
    Property(name="organization", data_type=DataType.TEXT),
    Property(name="source_url", data_type=DataType.TEXT),
    Property(name="document_type", data_type=DataType.TEXT),
    Property(name="ocr_derived", data_type=DataType.BOOL),
    Property(name="chunk_type", data_type=DataType.TEXT),
    Property(name="disease_name", data_type=DataType.TEXT),
    Property(name="disease_name_is_canonical", data_type=DataType.BOOL),
    Property(name="source_disease_label", data_type=DataType.TEXT),
    Property(name="section", data_type=DataType.TEXT),
    Property(name="page_start", data_type=DataType.INT),
    Property(name="page_end", data_type=DataType.INT),
    Property(name="word_count", data_type=DataType.INT),
    Property(name="chunk_index", data_type=DataType.INT),
    Property(name="chunk_total", data_type=DataType.INT),
    Property(name="is_inoculation_protocol", data_type=DataType.BOOL),
    Property(name="symptom", data_type=DataType.TEXT),
    Property(name="recommended_treatment", data_type=DataType.TEXT),
    Property(name="active_ingredient", data_type=DataType.TEXT),
    Property(name="dosage", data_type=DataType.TEXT),
    Property(name="crop_stage", data_type=DataType.TEXT),
]

In [27]:
SCHEMA_KEYS = {p.name for p in SCHEMA_PROPERTIES}

In [28]:
missing_from_schema = set(sample.keys()) - SCHEMA_KEYS


In [29]:

if missing_from_schema:
    print(f"WARNING: chunk fields not in Weaviate schema, will be dropped on ingest: {missing_from_schema}")
else:
    print("Schema covers every field in the sample chunk.")

if client.collections.exists(CLASS_NAME):
    print(f"Collection '{CLASS_NAME}' already exists - not recreating automatically.")
    print("If you need a clean rebuild, delete it explicitly first:")
    print(f'  client.collections.delete("{CLASS_NAME}")')
else:
    client.collections.create(
        name=CLASS_NAME,
        vectorizer_config=Configure.Vectorizer.none(),  # embeddings supplied externally
        properties=SCHEMA_PROPERTIES,
    )
    print(f"Created collection: {CLASS_NAME}")


Schema covers every field in the sample chunk.
Created collection: OryzaMindChunk


In [ ]:
# Batch Ingest

In [30]:
collection = client.collections.get(CLASS_NAME)


In [31]:

chunk_by_id = {c["chunk_id"]: c for c in chunks}

In [32]:

with collection.batch.dynamic() as batch:
    for i, chunk_id in enumerate(chunk_ids):
        chunk = chunk_by_id[chunk_id]
        # strict filter: only send fields the schema actually defines,
        # so a stray/renamed key fails loudly here rather than silently
        # ingesting garbage or erroring mid-batch
        properties = {k: v for k, v in chunk.items() if k in SCHEMA_KEYS}
        batch.add_object(
            properties=properties,
            vector=embeddings[i].tolist(),
        )

In [34]:
failed = collection.batch.failed_objects

In [35]:
if failed:
    print(f"WARNING: {len(failed)} objects failed to ingest. First failure:")
    print(failed[0])
else:
    print("Batch ingest completed with no reported failures.")


Batch ingest completed with no reported failures.


In [36]:

total_in_weaviate = collection.aggregate.over_all(total_count=True).total_count

In [37]:
print(f"\nTotal objects in Weaviate: {total_in_weaviate}")
print(f"Total chunks in source file: {len(chunks)}")
assert total_in_weaviate == len(chunks), (
    "MISMATCH - re-run count check before trusting this collection. "
    "Do not silently proceed to validation on a partial ingest."
)



Total objects in Weaviate: 2030
Total chunks in source file: 2030


In [38]:
from weaviate.classes.query import Filter

In [39]:

inoculation_tagged_treatment = collection.query.fetch_objects(
    filters=(
        Filter.by_property("is_inoculation_protocol").equal(True)
        & Filter.by_property("chunk_type").equal("treatment_record")
    ),
    limit=1,
).objects
assert len(inoculation_tagged_treatment) == 0, "Safety invariant broke on ingest - stop."
print("Post-ingest safety check: PASS (0 inoculation-protocol chunks as treatment_record)")

Post-ingest safety check: PASS (0 inoculation-protocol chunks as treatment_record)


In [40]:
print("\nCoverage per canonical disease class:")
for disease in sorted(CANONICAL_CLASSES):
    total = collection.query.fetch_objects(
        filters=Filter.by_property("disease_name").equal(disease), limit=1000,
    ).objects
    treatment_count = sum(1 for o in total if o.properties.get("chunk_type") == "treatment_record")
    print(f"  {disease}: {len(total)} chunks, {treatment_count} treatment_record")


Coverage per canonical disease class:
  Bacterial_Leaf_Blight: 292 chunks, 10 treatment_record
  Brown_Spot: 93 chunks, 5 treatment_record
  Leaf_Blast: 220 chunks, 39 treatment_record
  Narrow_Brown: 13 chunks, 1 treatment_record
  Rice_Tungro: 175 chunks, 19 treatment_record
  Sheath_Blight: 143 chunks, 28 treatment_record


In [ ]:
# Validation Queries

In [41]:

def search(query_text: str, top_k: int = 5, disease_filter: str = None):
    query_vec = model.encode(f"query: {query_text}", normalize_embeddings=True)

    kwargs = {"near_vector": query_vec.tolist(), "limit": top_k}
    if disease_filter:
        kwargs["filters"] = Filter.by_property("disease_name").equal(disease_filter)

    results = collection.query.near_vector(**kwargs)
    for obj in results.objects:
        p = obj.properties
        print(f"\n[{p.get('organization')}] {p.get('document_title')} "
              f"({p.get('chunk_type')}, disease={p.get('disease_name')}, "
              f"pages {p.get('page_start')}-{p.get('page_end')})")
        print(p.get("text", "")[:250])



In [42]:
print("=" * 70)
print("QUERY 1: treatment for Bacterial_Leaf_Blight")
search("treatment for Bacterial_Leaf_Blight")


QUERY 1: treatment for Bacterial_Leaf_Blight

[IRRI] Section 2. Bacterial diseases (figure_caption, disease=Bacterial Leaf Streak, pages 91-91)
BLS Fig. 4. Bacterial ooze (O) on streak lesions of leaf segments 14 days after inoculation with pure cul

[IRRI] Manual on Biotic Stress Resistance Evaluation, First Edition (2025) (narrative, disease=Bacterial Leaf Streak, pages 22-26)
Inoculation of Xoc using spraying method Materials Plants at maximum tillering stage (45-50 DAS) Spray bottles 100ml bacterial suspension inside ice chest Paper towels Nitrile or latex gloves Procedure 1.Inoculation is preferably done between 10:00 a

[IRRI] Section 2. Bacterial diseases (narrative, disease=Bacterial_Leaf_Blight, pages 3-7)
Bacterial blight syndrome exhibits three types of symptoms: leaf blight, kresek (the seedling blight or wilt phase), and the pale-yellow leaf. The disease has been referred to as "bacterial leaf blight" to indicate that the "leaf blight" phase of the

[University of Arkansas

In [43]:
print("\n" + "=" * 70)
print("QUERY 2: yellow spots on rice leaves")
search("yellow spots on rice leaves")


QUERY 2: yellow spots on rice leaves

[IRRI] Chapter 2. Foliar fungal diseases (figure_caption, disease=Brown_Spot, pages 2-2)
BSp Fig. 1. Brown spot of rice on the foliage.

[IRRI] Chapter 2. Foliar fungal diseases (figure_caption, disease=Brown_Spot, pages 2-2)
BSp Fig. 1. Brown spot on rice foliage.

[IRRI] Section 3. Virus and Phytoplasma Diseases (figure_caption, disease=None, pages 5-5)
RYD Fig. 1. A rice plant infected with yellow dwarf.

[IRRI] Chapter 2. Foliar fungal diseases (narrative, disease=None, pages 39-40)
The typical symptoms of RS are yellow-orange spots with streaks advancing toward the leaf tips (RS Figure 1).There is no clear margin on the lesions. Initial lesions are more apparent when observed from the abaxial side of the leaf. Four types of les

[IRRI] Manual on Biotic Stress Resistance Evaluation, First Edition (2025) (narrative, disease=Bacterial_Leaf_Blight, pages 6-8)
Perform a quick survey interview with the farmer to obtain the following information: (a

In [44]:
# 3. Cross-organization - check both IRRI and Arkansas can surface
print("\n" + "=" * 70)
print("QUERY 3: fungicide dosage for sheath blight")
search("fungicide dosage per acre for sheath blight")


QUERY 3: fungicide dosage for sheath blight

[University of Arkansas] Arkansas Rice Production Handbook — Management of Rice Diseases (treatment_record, disease=Sheath_Blight, pages 5-5)
Fungicides recommended for rice sheath blight, kernel smut and false smut control/suppression. For Sheath Blight: apply Quilt Xcel 2.2 EC (active ingredient azoxystrobin + propiconazole) at 14 - 27 fl oz per acre. Tested rates for Quilt Xcel were 17.

[University of Arkansas] Arkansas Rice Production Handbook — Management of Rice Diseases (treatment_record, disease=Sheath_Blight, pages 5-5)
Fungicides recommended for rice sheath blight, kernel smut and false smut control/suppression. For Sheath Blight: apply Stratego (active ingredient trifloxystrobin + propiconazole) at 16 - 19 fl oz per acre. Read and follow label application directio

[University of Arkansas] Arkansas Rice Production Handbook — Management of Rice Diseases (treatment_record, disease=Sheath_Blight, pages 5-5)
Fungicides recommended f

In [45]:
# 4. Narrow_Brown specifically - the known thin-coverage class
print("\n" + "=" * 70)
print("QUERY 4: how to manage Narrow_Brown leaf spot")
search("how to manage Narrow_Brown leaf spot", disease_filter="Narrow_Brown")



QUERY 4: how to manage Narrow_Brown leaf spot

[University of Arkansas] Arkansas Rice Production Handbook — Management of Rice Diseases (treatment_record, disease=Narrow_Brown, pages 13-14)
Although their appearances are similar, blanking from narrow brown leaf spot is minimal, and the described symptoms usually develop only near the completion of grain fill. Some spikelets and individual flowers may be blanked by this disease, especial

[University of Arkansas] Arkansas Rice Production Handbook — Management of Rice Diseases (narrative, disease=Narrow_Brown, pages 13-14)
Narrow brown leaf spot has been a minor, late-season disease of rice in Arkansas. Traditionally, it has little effect on yield or quality loss, but it caused problems late in 2006. Narrow brown leaf spot is caused by the fungus Cercospora oryzae and 

[IRRI] Chapter 2. Foliar fungal diseases (narrative, disease=Narrow_Brown, pages 20-20)
The initial symptom involves the presence of short, linear brown lesions, which a

In [46]:
# 5. Disease code recognition - IRRI chapters use codes (BSp, RBl, BB) not
# full names in running prose; this checks Phase 3's fix actually surfaces
print("\n" + "=" * 70)
print("QUERY 5: chemical control of brown spot disease")
search("chemical control of brown spot disease")


QUERY 5: chemical control of brown spot disease

[IRRI] Section 2. Bacterial diseases (treatment_record, disease=None, pages 3-3)
Agrochemicals such as kasugamycin, oxilinic acid, copper bactericide, Ag containing bactericide, and biocontrol agents, such as Trichoderma atroviride, have been registered and used in Japan (K. Aegami, pers. comm.). Because of its seedborne nature, 

[University of Arkansas] Arkansas Rice Production Handbook — Management of Rice Diseases (narrative, disease=Brown_Spot, pages 11-12)
On most cultivars, severe brown spot indicates a nutritional problem that fungicides cannot correct. Therefore, fungicide application to reduce disease incidence will not prevent yield reductions. Correction of the underlying nutritional problem(s) i

[IRRI] Section 2. Bacterial diseases (narrative, disease=Bacterial_Leaf_Blight, pages 31-32)
decades ago (Tagami and Mizukami 1962, Ou 1985). The Bordeaux mixture and copper and mercuric compounds did not show any success in contro